[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/polars-certified/notebooks/day-05-string-operations-dates-and-data-types.ipynb#scrollTo=30a1b2c3)

---
# Day 5 · String Operations, Dates, and Data Types
**certified-journeys / polars-certified** &nbsp;|&nbsp; Data Engineering

> **Goal for today:** Master Polars string and datetime namespaces, Categorical dtype, and schema/type management.

---
## The namespace model

Polars organises type-specific methods under **namespaces** accessed through the column expression:

| Namespace | Access | For |
|---|---|---|
| String | `pl.col("x").str.*` | `String` / `Utf8` columns |
| Datetime | `pl.col("x").dt.*` | `Date`, `Datetime`, `Duration` columns |
| List | `pl.col("x").list.*` | `List[T]` columns |
| Categorical | `pl.col("x").cat.*` | `Categorical` / `Enum` columns |

You get IDE autocomplete, type safety, and zero Python overhead — all namespace methods are compiled Rust.

> **Tip:** `Categorical` dtype can reduce memory 10x for low-cardinality string columns. Always use it for IDs and categories.

In [ ]:
%pip install -q polars numpy

---
## Step 1 · String namespace: `.str.*`

The `.str` namespace exposes all string operations as vectorized expressions —
no `.apply(lambda x: x.lower())`, no Python loops.

In [ ]:
import polars as pl

df = pl.DataFrame({
    "name":    ["Alice Smith", "BOB jones", "carol  WHITE", "dave BROWN", "Eve Taylor"],
    "email":   ["alice@corp.com", "bob@other.org", "carol@corp.com", "dave@corp.com", "eve@other.org"],
    "code":    ["PROD-001-A", "PROD-002-B", "SERV-003-A", "SERV-004-C", "PROD-005-B"],
    "notes":   ["ships fast", "out of stock; contact sales", "express only", None, "bulk discount available"],
})

result = df.with_columns(
    # Lowercase and uppercase
    pl.col("name").str.to_lowercase().alias("name_lower"),

    # Check if email contains a specific domain
    pl.col("email").str.contains("@corp.com").alias("is_internal"),

    # Replace substring (supports regex by default)
    pl.col("name").str.replace(r"\s+", " ").alias("name_normalized"),  # collapse extra spaces

    # Split on delimiter — returns a List[String] column
    pl.col("code").str.split("-").alias("code_parts"),

    # Extract with regex — capture group 1 is the category prefix
    pl.col("code").str.extract(r"^([A-Z]+)-", group_index=1).alias("code_type"),

    # Length in characters (use len_bytes for byte length — differs for Unicode)
    pl.col("name").str.len_chars().alias("name_len"),
)

print("String operations:")
print(result.select(
    pl.col("name", "name_lower", "name_normalized", "is_internal",
           "code", "code_parts", "code_type", "name_len")
))

**What just happened?**
- All `.str.*` methods are **vectorized** — they process every row in a single compiled Rust call
- `str.contains()` returns a **Boolean column** — directly usable in `filter()`
- `str.split("-")` returned a `List[String]` column — each cell is a list; use `.list.get(0)` to extract elements
- `str.extract()` with a regex **capture group** extracts the matched substring — `group_index=1` for first group
- **`str.len_chars()` vs `str.len_bytes()`** — for ASCII they're equal; for Unicode (e.g. emoji) they differ; use `len_chars()` for display length

---
## Step 2 · Parse dates from strings

Real-world data almost always has dates as strings. Polars provides `str.to_date()` and
`str.to_datetime()` to parse them without leaving the expression API.

In [ ]:
df_dates = pl.DataFrame({
    "date_iso":    ["2024-01-15", "2024-03-22", "2024-07-04", "2024-11-28"],
    "date_us":     ["01/15/2024", "03/22/2024", "07/04/2024", "11/28/2024"],
    "datetime_ts": ["2024-01-15 09:30:00", "2024-03-22 14:45:00",
                    "2024-07-04 08:00:00", "2024-11-28 17:15:00"],
})

parsed = df_dates.with_columns(
    # Parse ISO date string → Date dtype
    pl.col("date_iso").str.to_date("%Y-%m-%d").alias("date_parsed"),

    # Parse US-format date string → Date dtype
    pl.col("date_us").str.to_date("%m/%d/%Y").alias("date_us_parsed"),

    # Parse datetime string → Datetime dtype
    pl.col("datetime_ts").str.to_datetime("%Y-%m-%d %H:%M:%S").alias("datetime_parsed"),
)

print("Parsed date types:")
print(parsed.schema)
print()
print(parsed)

**What just happened?**
- `str.to_date("%Y-%m-%d")` parsed the ISO string into a Polars `Date` dtype — stored as int32 days since epoch
- The **format string** follows `strftime` conventions (`%Y` = 4-digit year, `%m` = month, `%d` = day)
- After parsing, the column dtype changed from `String` to `Date` — now all `.dt.*` methods are available
- `str.to_datetime()` produces a `Datetime` dtype that includes time components
- **Always parse dates early in your pipeline** — it unlocks efficient date arithmetic and filtering

---
## Step 3 · Datetime namespace: `.dt.*`

Once a column is a `Date` or `Datetime` dtype, the `.dt` namespace gives you all calendar extraction
methods, truncation, offset arithmetic, and duration math.

In [ ]:
from datetime import date

df_dt = pl.DataFrame({
    "event": ["launch", "review", "release", "followup", "audit"],
    "date":  [
        date(2024, 1, 15),
        date(2024, 3, 22),
        date(2024, 7, 4),
        date(2024, 11, 28),
        date(2025, 2, 14),
    ],
})

enriched = df_dt.with_columns(
    pl.col("date").dt.year().alias("year"),          # extract year
    pl.col("date").dt.month().alias("month"),         # extract month (1–12)
    pl.col("date").dt.day().alias("day"),             # extract day of month
    pl.col("date").dt.weekday().alias("weekday"),     # 1=Mon … 7=Sun

    # Truncate to start of month — useful for monthly groupings
    pl.col("date").dt.truncate("1mo").alias("month_start"),

    # Offset by 30 days — date arithmetic
    pl.col("date").dt.offset_by("30d").alias("due_date"),
)
print("Date extraction and arithmetic:")
print(enriched)

print()
# Date subtraction — result is a Duration dtype
df_dur = df_dt.with_columns(
    pl.col("date").alias("event_date"),
    pl.lit(date(2024, 1, 1)).cast(pl.Date).alias("base_date"),
).with_columns(
    (pl.col("event_date") - pl.col("base_date")).alias("days_since_start")
)
print("Date subtraction (Duration):")
print(df_dur.select(pl.col("event", "event_date", "days_since_start")))

**What just happened?**
- `.dt.year()`, `.dt.month()`, `.dt.day()` extract calendar components — all return `Int32` columns
- `.dt.weekday()` returns 1 (Monday) through 7 (Sunday) — useful for business day filtering
- **`.dt.truncate("1mo")`** rounds down to the first of the month — the string `"1mo"` is a Polars duration string (also: `"1d"`, `"1w"`, `"1y"`)
- `.dt.offset_by("30d")` adds 30 days — equivalent to `timedelta(days=30)` but vectorized
- **Subtracting two `Date` columns produces a `Duration`** dtype — representing a time interval, not an integer

---
## Step 4 · Categorical dtype — memory efficiency

`Categorical` dtype stores each unique string once in a dictionary, then uses small integer codes
for every row. For low-cardinality columns (e.g. country, status, product category) this reduces
memory dramatically and speeds up `group_by` operations.

In [ ]:
import random

random.seed(1)
n = 500_000
statuses = ["pending", "active", "closed", "cancelled"]
countries = ["US", "UK", "DE", "FR", "JP", "AU", "CA", "BR"]

# String DataFrame — baseline
df_str = pl.DataFrame({
    "id":      range(n),
    "status":  [random.choice(statuses)  for _ in range(n)],
    "country": [random.choice(countries) for _ in range(n)],
})

# Categorical DataFrame — converted
df_cat = df_str.with_columns(
    pl.col("status").cast(pl.Categorical),    # String → Categorical
    pl.col("country").cast(pl.Categorical),   # String → Categorical
)

size_str = df_str.estimated_size("mb")
size_cat = df_cat.estimated_size("mb")

print(f"String DataFrame size:      {size_str:.2f} MB")
print(f"Categorical DataFrame size: {size_cat:.2f} MB")
print(f"Memory reduction:           {(1 - size_cat/size_str)*100:.1f}%")
print()
print("Categorical dtype in schema:")
print(df_cat.schema)

**What just happened?**
- `.cast(pl.Categorical)` converted string columns to integer-coded dictionaries — typically 5–15x smaller
- `df.estimated_size("mb")` returns the in-memory size in megabytes — use this to measure dtype impact
- **`group_by` on `Categorical` columns is faster** than on `String` — integer comparison instead of string comparison
- The unique values (dictionary) are stored once — adding 1 million more rows of the same values adds almost no extra memory
- **Use Categorical for any column with fewer than ~1000 unique values** relative to the total row count

---
## Step 5 · Enum dtype vs Categorical

| | `Categorical` | `Enum` |
|---|---|---|
| Values | Discovered at runtime | Fixed at schema definition time |
| New values | Auto-added to dictionary | Raises an error |
| Ordering | Unordered | Can be ordered |
| Use when | Cardinality unknown at code time | Fixed set (e.g. status codes, severity levels) |

In [ ]:
# Enum dtype — fixed vocabulary defined at schema time
severity_type = pl.Enum(["low", "medium", "high", "critical"])  # fixed ordered categories

df_enum = pl.DataFrame({
    "ticket_id": [1, 2, 3, 4, 5],
    "severity":  ["low", "high", "medium", "critical", "low"],
}).with_columns(
    pl.col("severity").cast(severity_type)   # cast to Enum
)

print("Enum dtype:")
print(df_enum)
print("Schema:", df_enum.schema)
print()

# Enum enforces the vocabulary — uncommenting the next block would raise InvalidOperationError
# df_bad = pl.DataFrame({"severity": ["urgent"]}).with_columns(
#     pl.col("severity").cast(severity_type)  # "urgent" is not in Enum → error
# )

# Sorting with Enum uses the declared order (low < medium < high < critical)
print("Sorted by Enum order:")
print(df_enum.sort("severity"))

**What just happened?**
- `pl.Enum([...])` defines a **fixed vocabulary** — any value outside the list will raise an error at cast time
- Sorting an `Enum` column uses the **declared order** of the category list, not alphabetical — `low` sorts before `high`
- **Use `Enum` when the set of values is known and fixed** (status codes, severity, day-of-week, etc.) — it acts as a schema contract
- Use `Categorical` when the set of values is discovered at runtime or can grow
- Both `Enum` and `Categorical` use integer encoding internally — same memory efficiency

---
## Step 6 · Schema inspection and type coercion

Polars schemas are strict — mismatched types cause errors at pipeline definition time, not at
runtime surprises. Learn how to inspect, cast, and override types when reading data.

In [ ]:
import csv

# Write a CSV with mixed types that need careful handling
csv_path = "/tmp/mixed_types.csv"
with open(csv_path, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["id", "score", "flag", "amount"])
    w.writerows([
        ["001", "87.5", "1", "1200.50"],
        ["002", "92.0", "0", "980.00"],
        ["003", "75.3", "1", "2100.75"],
    ])

# --- Default read — Polars infers types ---
df_inferred = pl.read_csv(csv_path)
print("Inferred schema:", df_inferred.schema)
print(df_inferred)
print()

# --- schema_overrides — tell Polars the correct types at read time ---
df_typed = pl.read_csv(
    csv_path,
    schema_overrides={
        "id":     pl.String,    # keep as string (leading zero matters)
        "score":  pl.Float32,   # use Float32 instead of Float64 to save memory
        "flag":   pl.Boolean,   # read 0/1 integers as Boolean
        "amount": pl.Float64,
    }
)
print("Overridden schema:", df_typed.schema)
print(df_typed)
print()

# --- Post-read casting with df.cast() ---
df_cast = df_inferred.cast({
    "id":    pl.String,
    "score": pl.Float32,
})
print("Post-read cast schema:", df_cast.schema)
print(df_cast)

**What just happened?**
- By default Polars inferred `id` as `Int64` — but leading zeros would be lost if the ID were `"007"`
- `schema_overrides` overrides the inferred type **during the read** — more efficient than read-then-cast
- `flag` was read as `Int64` by default — `schema_overrides` cast it to `Boolean` at source
- `df.cast({"col": dtype})` applies a type cast to named columns post-read — useful for adjusting after `scan_csv()`
- **`Float32` vs `Float64`** — use `Float32` when 7-digit precision is sufficient; halves the column memory

---
## Challenge — do this yourself

The DataFrame below has messy date strings. Your tasks:
1. Parse the `raw_date` column into a proper `Date` dtype (handle mixed formats using `try_parse=True` or a single format)
2. Extract the **week number** (`.dt.week()`) and **quarter** (derive from `.dt.month()` using `pl.when()`)
3. Cast the `category` column to `Categorical`
4. Print the final schema and result

In [ ]:
# Your solution here
df_messy = pl.DataFrame({
    "event_id":  [1, 2, 3, 4, 5, 6],
    "raw_date":  ["2024-03-15", "2024-07-22", "2024-01-08",
                  "2024-10-31", "2024-05-14", "2024-12-25"],
    "category":  ["sale", "return", "sale", "exchange", "sale", "return"],
    "amount":    [120.5, -45.0, 890.0, -200.0, 55.25, -30.0],
})

result = (
    df_messy
    # TODO: parse raw_date to Date dtype using str.to_date()
    # TODO: add week_number column using .dt.week()
    # TODO: add quarter column: Q1 (months 1-3), Q2 (4-6), Q3 (7-9), Q4 (10-12)
    #        use pl.when().when().when().otherwise() on .dt.month()
    # TODO: cast category to Categorical
)
print(result)
# print(result.schema)

---
## Day 5 key concepts recap

| Concept | What to remember |
|---|---|
| `.str.*` namespace | All string operations: `to_lowercase`, `contains`, `replace`, `split`, `extract`, `len_chars` |
| `str.to_date()` / `str.to_datetime()` | Parse string columns to `Date` / `Datetime` dtype using format strings |
| `.dt.*` namespace | Calendar extraction (`year`, `month`, `weekday`), `truncate`, `offset_by`, date arithmetic |
| `Categorical` | Cast low-cardinality strings — 5–15x memory reduction, faster `group_by` |
| `Enum` | Fixed vocabulary — schema contract, ordered sorting, error on unknown values |
| `schema_overrides` | Override inferred types at `read_csv` / `scan_csv` time — most efficient approach |
| `df.cast({})` | Post-read type casting by column name — use after inferred reads |

> **Tip:** `Categorical` dtype can reduce memory 10x for low-cardinality string columns. Always use it for status codes, country codes, and product categories in production pipelines.

---
## What's next

You've completed the 5-day Polars core track. You now have the mental model, expression API, lazy evaluation, analytics patterns, and data type management to build production Polars pipelines.

Mark Day 5 complete in your [tracker](../index.html).